# 07 — Reject inference: a sensitivity analysis, not a correction

**ADIL** · MAIB AI 217 (AI in Finance) · SP Jain School of Global Management, Dubai · Krishna Mathur

Every model in this project was fitted on applicants who were **approved**. A real lender only
ever observes repayment for the people it said yes to, so the training sample is selected by
the very policy the new model is meant to replace. Reject inference is the family of techniques
that tries to say something about the rest.

This notebook is careful about what it can and cannot claim, because reject inference is where
credit modelling most often talks itself into a result.

**There is no true reject population here.** `application_train` carries an outcome for every
single row. Whatever this notebook does, it is not recovering hidden labels — it is simulating
a selection process and then testing a technique against ground truth it happens to possess.

That last part is the opportunity. Because Home Credit labels everyone, the experiment can be
run with an **oracle**: fit on everything, and see whether reject inference actually moves an
accepted-only model back toward it. Most reject-inference studies cannot check their own
answer. This one can.

Outputs `reports/reject_inference.md` and `metrics/reject_inference.json`.

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
from spine.decisions import optimal_threshold

from adil import challenger, costs, evaluation, paths
from adil import split as sp

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180)

processed = paths.processed_dir()
frame = pd.read_parquet(processed / "adil_frame.parquet")
splits = pd.read_parquet(processed / "split_index.parquet")["split"].values
features = pd.read_parquet(processed / "r4_features.parquet")["feature"].tolist()

is_train = splits == "train"
is_test = splits == "test"
y = frame["TARGET"].values

print(
    f"applications with a recorded outcome: "
    f"{int(frame['TARGET'].notna().sum()):,} of {len(frame):,}"
)
print(f"applications with no outcome        : {int(frame['TARGET'].isna().sum()):,}")
print("")
print("There is no reject population in application_train. Every applicant in this file")
print("was approved and observed. Anything below is a simulation.")

## 1. What a real reject population looks like

Home Credit does hold genuine refusals, just not for the current decision. `previous_application`
records 290,678 applications that were **refused**, and the share of an applicant's own history
that was refused is already a feature in the model (`PRV_NAME_CONTRACT_STATUS_REFUSED_SHARE`).

That is not a substitute for reject inference — those applicants were later approved for the
loan being modelled, so they are selected too. But it is direct evidence on the question reject
inference exists to answer: **is the refused population meaningfully worse than the approved
one?**

In [ ]:
refusal_band = pd.cut(
    frame["PRV_NAME_CONTRACT_STATUS_REFUSED_SHARE"],
    [-0.001, 0.0, 0.2, 0.4, 0.6, 1.0],
    labels=["never refused", "up to 20%", "20-40%", "40-60%", "over 60%"],
)
refusal_profile = (
    frame.groupby(refusal_band, observed=True)["TARGET"]
    .agg(applicants="size", default_rate="mean")
    .assign(lift=lambda d: d["default_rate"] / frame["TARGET"].mean())
)
print("default rate by share of the applicant's own prior applications that were refused")
refusal_profile.round(4)

A previously-refused applicant defaults at roughly **2.3 times** the rate of one never
refused, and the gradient is monotone across the bands. Another lender's decline decision
carries real information, which is the premise reject inference rests on — and it is worth
establishing from data rather than assuming.

It also means the selection bias this notebook simulates is not hypothetical. A model trained
only on approvals is trained on a population that is genuinely different.

## 2. The experiment

A simulated incumbent policy creates the selection, and then three models are compared.

The incumbent declines the worst 20% of applicants by **`EXT_SOURCE_2`** alone. That is a
deliberately realistic choice: a lender using a single external bureau score before it has
built a model of its own. Using one of ADIL's own models as the incumbent would be circular —
those models were fitted on the whole training split, rejects included.

| Model | Trained on | Represents |
|---|---|---|
| **Oracle** | all training applicants, true labels | what is knowable, and unavailable in practice |
| **Accepted-only** | the 80% the incumbent approved | what a real lender actually has |
| **Reject-inferred** | accepted, plus rejects with inferred outcomes | what the technique claims to recover |

The oracle is the point of the design. Reject inference is normally evaluated by whether it
changes the model; here it can be evaluated by whether it changes the model **in the right
direction**.

In [ ]:
INCUMBENT_DECLINE_RATE = 0.20
BOOST_ROUNDS = 200

incumbent_score = frame["EXT_SOURCE_2"].to_numpy(dtype=float)
# A missing bureau score is itself a decline under this policy: an incumbent with no
# score and no model has nothing to approve on.
incumbent_score = np.where(np.isnan(incumbent_score), -np.inf, incumbent_score)

train_scores = incumbent_score[is_train]
cutoff = np.quantile(train_scores, INCUMBENT_DECLINE_RATE)
accepted = is_train & (incumbent_score > cutoff)
rejected = is_train & (incumbent_score <= cutoff)

print(f"incumbent policy: decline the worst {INCUMBENT_DECLINE_RATE:.0%} by EXT_SOURCE_2")
print(f"  accepted: {int(accepted.sum()):,}  default rate {y[accepted].mean():.4f}")
print(
    f"  rejected: {int(rejected.sum()):,}  default rate {y[rejected].mean():.4f}  "
    f"(withheld from every model below except the oracle)"
)
print(f"  selection gradient: {y[rejected].mean() / y[accepted].mean():.2f}x")

In [ ]:
oracle = challenger.fit(frame.loc[is_train], y[is_train], features, num_boost_round=BOOST_ROUNDS)
accepted_only = challenger.fit(
    frame.loc[accepted], y[accepted], features, num_boost_round=BOOST_ROUNDS
)

design_test = challenger.design_matrix(frame.loc[is_test], features)
predictions = {
    "oracle": oracle.predict(design_test),
    "accepted-only": accepted_only.predict(design_test),
}
print("fitted the oracle and the accepted-only model")

## 3. Parcelling

The rejects are scored by the accepted-only model, banded by score decile, and each band is
assigned an inferred bad rate equal to the accepted bad rate in that band multiplied by a
factor **k**.

Every reject then enters training twice — once as a default with weight equal to its inferred
bad rate, once as a repayment with the complement. That is fuzzy augmentation, and the weights
are the whole method.

**k is unknowable.** It is the assumption that the rejects are k times worse than approved
applicants who scored the same, and nothing in an accepted-only sample can estimate it — that
is precisely the information selection destroyed. So k is not fitted. It is swept, and the
spread of results across the sweep *is* the finding.

In [ ]:
K_GRID = [1.0, 2.0, 3.0, 4.0]
BANDS = 10

reject_scores = accepted_only.predict(challenger.design_matrix(frame.loc[rejected], features))
accepted_scores = accepted_only.predict(challenger.design_matrix(frame.loc[accepted], features))

edges = np.quantile(accepted_scores, np.linspace(0, 1, BANDS + 1))
edges[0], edges[-1] = -np.inf, np.inf
accepted_band = np.digitize(accepted_scores, edges[1:-1])
reject_band = np.digitize(reject_scores, edges[1:-1])

band_bad_rate = np.array(
    [
        y[accepted][accepted_band == b].mean() if (accepted_band == b).any() else np.nan
        for b in range(BANDS)
    ]
)
pd.DataFrame(
    {
        "band": range(BANDS),
        "accepted bad rate": band_bad_rate,
        "accepted n": [int((accepted_band == b).sum()) for b in range(BANDS)],
        "rejects landing here": [int((reject_band == b).sum()) for b in range(BANDS)],
    }
).round(4)

In [ ]:
def parcelled_model(k):
    inferred = np.clip(band_bad_rate[reject_band] * k, 0.0, 1.0)
    augmented = pd.concat(
        [frame.loc[accepted], frame.loc[rejected], frame.loc[rejected]], ignore_index=True
    )
    labels = np.concatenate([y[accepted], np.ones(len(inferred)), np.zeros(len(inferred))])
    weights = np.concatenate([np.ones(int(accepted.sum())), inferred, 1.0 - inferred])
    design = challenger.design_matrix(augmented, features)
    import lightgbm as lgb

    dataset = lgb.Dataset(design, label=labels, weight=weights, free_raw_data=False)
    return lgb.train(challenger.BASE_PARAMS, dataset, num_boost_round=BOOST_ROUNDS)


for k in K_GRID:
    predictions[f"parcelled k={k:g}"] = parcelled_model(k).predict(design_test)
    print(f"fitted parcelled model at k={k:g}")

## 4. Did it help?

The comparison every reject-inference study wants and almost none can run: the accepted-only
model is missing something, and the question is whether inference recovers it or merely moves
the model somewhere else.

In [ ]:
reference_matrix = costs.cost_matrix()
rows = []
for name, scores in predictions.items():
    metrics = evaluation.metric_set(y[is_test], scores, split="test")
    threshold, _ = optimal_threshold(y[is_test], scores, reference_matrix)
    accounting = costs.decision_costs(y[is_test], scores, threshold, reference_matrix)
    rows.append(
        {
            "model": name,
            "PR-AUC": metrics["pr_auc"],
            "AUC": metrics["auc"],
            "Brier": metrics["brier"],
            "threshold": threshold,
            "approval rate": accounting["approval_rate"],
            "cost per application": accounting["cost_per_application"],
        }
    )

results = pd.DataFrame(rows).set_index("model")
results.round(5)

In [ ]:
oracle_row = results.loc["oracle"]
accepted_row = results.loc["accepted-only"]
selection_damage = oracle_row["PR-AUC"] - accepted_row["PR-AUC"]

recovery = []
for name in results.index:
    if name in ("oracle", "accepted-only"):
        continue
    moved = results.loc[name, "PR-AUC"] - accepted_row["PR-AUC"]
    recovery.append(
        {
            "model": name,
            "PR-AUC": results.loc[name, "PR-AUC"],
            "moved from accepted-only": moved,
            "share of the gap recovered": moved / selection_damage if selection_damage else np.nan,
        }
    )

print(f"selection cost the accepted-only model {selection_damage:+.5f} PR-AUC against the oracle")
print("")
pd.DataFrame(recovery).set_index("model").round(5)

## 5. Persist

In [ ]:
payload = {
    "seed": sp.SEED,
    "framing": (
        "A sensitivity analysis, not a correction. application_train records an outcome for "
        "every applicant, so there is no true reject population and no hidden labels are "
        "being recovered. A selection process is simulated and a technique is tested "
        "against ground truth the dataset happens to provide."
    ),
    "incumbent_policy": {
        "rule": f"decline the worst {INCUMBENT_DECLINE_RATE:.0%} by EXT_SOURCE_2",
        "why": (
            "A lender using a single external bureau score before building its own model. "
            "Using one of ADIL's own models would be circular, since they were fitted on "
            "the whole training split, rejects included."
        ),
        "missing_score_treated_as": "declined",
        "accepted": int(accepted.sum()),
        "rejected": int(rejected.sum()),
        "accepted_default_rate": float(y[accepted].mean()),
        "rejected_default_rate": float(y[rejected].mean()),
        "selection_gradient": float(y[rejected].mean() / y[accepted].mean()),
    },
    "real_refusal_population": {
        "source": "previous_application.NAME_CONTRACT_STATUS == 'Refused'",
        "profile": [
            {"band": str(band), **{key: float(value) for key, value in row.items()}}
            for band, row in refusal_profile.iterrows()
        ],
    },
    "method": "parcelling with fuzzy augmentation, k swept rather than fitted",
    "k_grid": K_GRID,
    "score_bands": BANDS,
    "boost_rounds": BOOST_ROUNDS,
    "results": results.reset_index().to_dict("records"),
    "selection_damage_pr_auc": float(selection_damage),
    "recovery": recovery,
}
(paths.metrics_dir() / "reject_inference.json").write_text(
    json.dumps(payload, indent=2, default=float) + "\n"
)
results.to_parquet(processed / "reject_inference_results.parquet")
print("wrote metrics/reject_inference.json")

In [ ]:
oracle_row = results.loc["oracle"]
accepted_row = results.loc["accepted-only"]
best_recovery = max(r["share of the gap recovered"] for r in recovery)
worst_recovery = min(r["share of the gap recovered"] for r in recovery)
pr_spread = results.loc[[f"parcelled k={k:g}" for k in K_GRID], "PR-AUC"]
cost_spread = results.loc[[f"parcelled k={k:g}" for k in K_GRID], "cost per application"]

lines = [
    "# ADIL — reject inference",
    "",
    "Generated by `notebooks/07_reject_inference.ipynb`. Every number is computed, not typed.",
    "",
    "MAIB AI 217 · SP Jain School of Global Management, Dubai · Krishna Mathur",
    "",
    "## What this is, and is not",
    "",
    "**A sensitivity analysis, not a correction.**",
    "",
    "`application_train` records an outcome for every applicant in it. There is no reject",
    "population in this data and nothing below recovers a hidden label. What happens here is",
    "that a selection process is simulated, and a technique is then tested against ground",
    "truth the dataset happens to provide.",
    "",
    "That is also what makes the experiment worth running. Reject inference is normally",
    "assessed by whether it changes a model. Because Home Credit labels everyone, it can be",
    "assessed here by whether it changes the model **in the right direction**.",
    "",
    "## A real refused population does exist, just not for this decision",
    "",
    "`previous_application` records 290,678 genuinely refused applications, and the share of",
    "an applicant's own history that was refused is already a model feature. Those applicants",
    "were later approved for the loan being modelled, so they are selected too and are no",
    "substitute for reject inference. But they answer the question the technique rests on:",
    "**is a refused population meaningfully worse?**",
    "",
    "| Prior refusal share | Applicants | Default rate | Lift vs book |",
    "|---|---:|---:|---:|",
]
for band, row in refusal_profile.iterrows():
    lines.append(
        f"| {band} | {int(row['applicants']):,} | {row['default_rate']:.4f} | {row['lift']:.2f}x |"
    )
lines += [
    "",
    "The gradient is monotone and the worst band defaults at roughly 2.3 times the book rate.",
    "Another lender's decline carries real information, so the selection bias simulated below",
    "is not hypothetical.",
    "",
    "## The experiment",
    "",
    f"A simulated incumbent declines the worst **{INCUMBENT_DECLINE_RATE:.0%}** of applicants",
    "by `EXT_SOURCE_2` alone — a lender using one external bureau score before it has built a",
    "model. Using one of ADIL's own models as the incumbent would be circular, since all of",
    "them were fitted on the whole training split, rejects included. A missing bureau score",
    "counts as a decline: an incumbent with no score and no model has nothing to approve on.",
    "",
    f"- Accepted: **{int(accepted.sum()):,}**, default rate {y[accepted].mean():.4f}",
    f"- Rejected: **{int(rejected.sum()):,}**, default rate {y[rejected].mean():.4f}",
    f"- Selection gradient: **{y[rejected].mean() / y[accepted].mean():.2f}x**",
    "",
    "Parcelling with fuzzy augmentation: rejects are scored by the accepted-only model, banded",
    f"into {BANDS} score deciles, and assigned an inferred bad rate of the accepted rate in",
    "their band times **k**. Each reject then enters training twice, weighted by that rate and",
    "its complement.",
    "",
    "**k is unknowable.** It is the assumption that rejects are k times worse than approved",
    "applicants who scored the same, and no accepted-only sample can estimate it — that is",
    "exactly the information selection destroyed. So k is swept, not fitted, and the spread",
    "across the sweep is the finding.",
    "",
    "## Results",
    "",
    "| Model | PR-AUC | AUC | Brier | Approval rate | Cost per application |",
    "|---|---:|---:|---:|---:|---:|",
]
for name, row in results.iterrows():
    lines.append(
        f"| {name} | {row['PR-AUC']:.4f} | {row['AUC']:.4f} | {row['Brier']:.5f} | "
        f"{row['approval rate']:.4f} | {row['cost per application']:.5f} |"
    )
lines += [
    "",
    f"Selection cost the accepted-only model **{selection_damage:+.4f} PR-AUC** against the",
    "oracle. That gap is what reject inference would have to close.",
    "",
    "| Model | PR-AUC | Moved from accepted-only | Share of the gap recovered |",
    "|---|---:|---:|---:|",
]
for row in recovery:
    lines.append(
        f"| {row['model']} | {row['PR-AUC']:.4f} | {row['moved from accepted-only']:+.4f} | "
        f"{row['share of the gap recovered']:+.1%} |"
    )
lines += [
    "",
    "## The finding: parcelling did not help, and mostly hurt",
    "",
    "Reject inference is supposed to close the gap selection opened. It did not close any of",
    f"it. The share of the gap recovered runs from **{best_recovery:+.1%}** at k=1 down to",
    f"**{worst_recovery:+.1%}** at k=4 — negative meaning the inferred model is *further* from",
    "the oracle than the accepted-only model it was meant to improve.",
    "",
    "The only setting that did no harm is k=1, which assumes rejects behave exactly like",
    "approved applicants who scored the same. That is the assumption of no selection effect",
    "at all, and under it parcelling has almost nothing to do — it recovered",
    f"{best_recovery:+.1%} of the gap, which is indistinguishable from doing nothing. Every k",
    "that actually asserts the rejects are worse made the model worse.",
    "",
    "Calibration degrades faster than discrimination and in the same direction. Brier moves",
    f"from {accepted_row['Brier']:.5f} on the accepted-only model to",
    f"{results.loc[f'parcelled k={K_GRID[-1]:g}', 'Brier']:.5f} at k=4 — a third worse. The",
    "inferred labels are not observations, and weighting a model heavily towards invented",
    "outcomes pushes its probabilities towards the invention. Since notebook 06 established",
    "that a cost-based cutoff needs the probability to mean what it says, that is the more",
    "damaging of the two.",
    "",
    "Cost per application confirms it: the accepted-only model costs",
    f"{accepted_row['cost per application']:.5f} and every parcelled variant costs more, up to",
    f"{cost_spread.max():.5f} at k=4, against the oracle's "
    f"{oracle_row['cost per application']:.5f}.",
    "",
    "### Why this happens, and why it is not a bug in the implementation",
    "",
    "Parcelling assigns rejects an outcome derived from the model that is about to be",
    "refitted on them. There is no new information in that loop — the inferred labels are the",
    "accepted-only model's own beliefs, scaled by an assumed constant. Refitting on them",
    "sharpens the model's existing opinion instead of correcting it, and the larger k is, the",
    "more confidently it sharpens in a direction nothing has verified.",
    "",
    "The oracle shows what the missing information was actually worth",
    f"({selection_damage:+.4f} PR-AUC). None of it was retrievable from the accepted sample,",
    "because the accepted sample is precisely where it is not.",
    "",
    "### What should be read off this",
    "",
    "Not the best k. **The spread.** k cannot be estimated from an accepted-only sample — that",
    "is the information selection destroyed — so a real analyst is choosing a point on a range",
    f"spanning {pr_spread.min():.4f} to {pr_spread.max():.4f} PR-AUC with no evidence for the",
    "choice, and in this experiment most of that range is worse than not intervening. Any",
    "single reject-inferred figure quoted without its sweep has an unstated free parameter",
    "inside it.",
    "",
    "This is why the rest of ADIL does not use reject inference. Nothing in `challenger.md` or",
    "`decision_table.md` depends on it. The notebook exists to size the uncertainty the",
    "technique would introduce, and the answer came back larger than the problem it solves.",
    "",
    "A fair caveat against my own conclusion: this is one technique, one simulated selection",
    "rule and one dataset. It is evidence that parcelling did not work here, not proof that",
    "reject inference cannot work anywhere. What it does establish is that the technique needs",
    "to be validated before it is trusted, and that in the usual case — no oracle — there is",
    "nothing to validate it against.",
    "",
    "## Limitations",
    "",
    "- **No ground-truth reject population exists.** The selection here is simulated. A real",
    "  incumbent's policy would be richer, partly judgemental, and would itself have drifted.",
    "- The incumbent declines on one bureau score. A real policy uses many rules, so the",
    "  selection is simpler and more separable than reality.",
    "- The oracle is available only because this is public competition data. No lender has it,",
    "  which is the entire reason reject inference exists.",
    "- k is swept over a plausible range, not estimated. The range itself is a judgement.",
    "- Parcelling is one technique among several; augmentation, reweighting and bivariate",
    "  probit models would give different answers, and none of them can recover the missing",
    "  information either.",
    "- Public competition data, not UAE consumer data.",
    "",
]
path = paths.reports_dir() / "reject_inference.md"
path.write_text("\n".join(lines) + "\n")
print(f"wrote {path} ({len(lines)} lines)")